## Homework 6: Decision Trees and Ensemble Learning

This is the personal implementation of the homework 6 of [ML Zoomcamp 2026](https://github.com/DataTalksClub/machine-learning-zoomcamp).

In [2]:
# get the data
!wget https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv

--2026-11-01 21:40:00--  https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 635180 (620K) [text/plain]
Saving to: ‘car_fuel_efficiency_2026.csv’

car_fuel_efficiency 100%[===================>] 620.29K  3.43MB/s    in 0.2s    

2026-11-01 21:40:01 (3.43 MB/s) - ‘car_fuel_efficiency_2026.csv’ saved [635180/635180]



In [1]:
# import required libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer

# load the dataset
df = pd.read_csv("car_fuel_efficiency_2026.csv")

# inspect the dataset
print("Shape:", df.shape)
df.head()

Shape: (10000, 11)


,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [2]:
# check missing values for each column
df.isnull().sum()

model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors                0
engine_displacement      0
num_cylinders            0
horsepower             877
vehicle_weight           0
acceleration           264
fuel_efficiency_mpg      0
dtype: int64

In [3]:
# fill missing values with zeros
df = df.fillna(0)
df.isnull().sum()

model_year             0
origin                 0
fuel_type              0
drivetrain             0
num_doors              0
engine_displacement    0
num_cylinders          0
horsepower             0
vehicle_weight         0
acceleration           0
fuel_efficiency_mpg    0
dtype: int64

In [4]:
# split the dataset
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

# check the dimensions of each split
len(df_train), len(df_val), len(df_test)

(6000, 2000, 2000)

In [5]:
# reset index for each
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

# extract the target variable from each split
y_train = df_train.fuel_efficiency_mpg.values
y_val = df_val.fuel_efficiency_mpg.values
y_test = df_test.fuel_efficiency_mpg.values

# remove the target column from the feature dataframes
del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']

# get the size of each set
print("Total size:", len(df))
print(f"Train, Val, Test: ({len(df_train)}, {len(df_val)}, {len(df_test)})")
print(f"y_train, y_val, y_test: ({len(y_train)}, {len(y_val)}, {len(y_test)})")

Total size: 10000
Train, Val, Test: (6000, 2000, 2000)
y_train, y_val, y_test: (6000, 2000, 2000)


In [6]:
# convert feature dataframes to lists of dictionaries
train_dicts = df_train.to_dict(orient='records')
val_dicts = df_val.to_dict(orient='records')
test_dicts = df_test.to_dict(orient='records')

# encode categorical features and construct sparse feature matrices
dv = DictVectorizer(sparse=True)

X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)
X_test = dv.transform(test_dicts)

# inspect the resulting matrix dimensions
X_train.shape, X_val.shape, X_test.shape

((6000, 16), (2000, 16), (2000, 16))

#### Q1. Which original feature is used for splitting the data?

Train a decision tree regressor with `max_depth=1` and identify the original feature used to split the data.

In [7]:
from sklearn.tree import DecisionTreeRegressor, export_text

# train a decision tree regressor with a maximum depth of 1
dt = DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train)

# display the tree's splitting rules
print(export_text(dt, feature_names=list(dv.get_feature_names_out())))

|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]



**Answer: `model_year`**

The decision tree selects `model_year` as its splitting feature. Cars with a model year up to 1997 receive a predicted fuel efficiency of approximately 28.61 MPG, while newer cars receive approximately 31.17 MPG.

#### Q2. What's the RMSE of this model on the validation data?

In [8]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

# train a random forest regressor with 10 trees
rf = RandomForestRegressor(
    n_estimators=10,
    random_state=1,
    n_jobs=-1
)

rf.fit(X_train, y_train)

# predict fuel efficiency on the validation set
y_pred = rf.predict(X_val)

# calculate validation RMSE
rmse = root_mean_squared_error(y_val, y_pred)

print(f"Validation RMSE: {rmse:.3f}")

Validation RMSE: 1.833


#### Q3. Which value gives the lowest validation RMSE?

In [9]:
n_estimators = [10, 50, 100, 150]

# evaluate random forests with different numbers of trees
scores = []

for n in [10, 50, 100, 150]:
    rf = RandomForestRegressor(
        n_estimators=n,
        random_state=1,
        n_jobs=-1
    )

    rf.fit(X_train, y_train)

    y_pred = rf.predict(X_val)
    rmse = root_mean_squared_error(y_val, y_pred)

    scores.append((n, rmse))

# display results
df_scores = pd.DataFrame(scores, columns=['n_estimators', 'rmse'])
df_scores

,n_estimators,rmse
0,10,1.833220
1,50,1.773988
2,100,1.769786
3,150,1.769198


**Answer: `150`**

The random forest with 150 trees achieved the lowest validation RMSE (1.769). Increasing the number of trees generally improved performance, although the improvement became marginal after 100 trees.

#### Q4. What's the best max_depth, using the mean RMSE?

In [10]:
# evaluate random forests with different maximum depths and numbers of trees
scores = []

for depth in [10, 15, 20, 25]:
    for n in [10, 50, 100, 150]:
        rf = RandomForestRegressor(
            n_estimators=n,
            max_depth=depth,
            random_state=1,
            n_jobs=-1
        )

        rf.fit(X_train, y_train)
        y_pred = rf.predict(X_val)
        rmse = root_mean_squared_error(y_val, y_pred)

        scores.append((depth, n, rmse))

# create a dataframe with the results
df_scores = pd.DataFrame(
    scores, columns=['max_depth', 'n_estimators', 'rmse']
)

# calculate the mean RMSE for each maximum depth
df_scores.groupby('max_depth')['rmse'].mean().sort_values()

max_depth
10    1.754392
15    1.784224
25    1.786305
20    1.786771
Name: rmse, dtype: float64

**Answer: `10`**

A maximum depth of 10 achieved the lowest mean validation RMSE (1.754). Increasing the depth beyond 10 did not improve validation performance, suggesting that deeper trees may be capturing unnecessary complexity.

#### Q5. What's the most important feature (among these 4)?

In [11]:
# train a random forest for feature importance analysis
rf = RandomForestRegressor(
    n_estimators=10,
    max_depth=20,
    random_state=1,
    n_jobs=-1
)

rf.fit(X_train, y_train)

# extract feature names and their importance scores
feature_names = dv.get_feature_names_out()
importances = rf.feature_importances_

# create a dataframe sorted by feature importance
df_importance = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False)

df_importance

,feature,importance
9,model_year,0.296107
15,vehicle_weight,0.203313
6,fuel_type=Gasoline,0.160712
8,horsepower,0.078271
4,engine_displacement,0.061281
0,acceleration,0.054998
14,origin=USA,0.042152
2,drivetrain=Front-wheel drive,0.020062
11,num_doors,0.019401
5,fuel_type=Diesel,0.016571


**Answer: `vehicle_weight`**

Among the four given features, `vehicle_weight` has the highest importance score (0.2033). Although `model_year` has the highest importance overall (0.2961), it is not among the answer options.

#### Q6. Which eta leads to the best RMSE score on the validation dataset?

In [13]:
# check whether XGBoost is installed and display its version
import xgboost as xgb

print("XGBoost version:", xgb.__version__)

XGBoost version: 3.4.1


In [14]:
# prepare the training and validation datasets for XGBoost
features = dv.get_feature_names_out().tolist()

dtrain = xgb.DMatrix(
    X_train,
    label=y_train,
    feature_names=features
)

dval = xgb.DMatrix(
    X_val,
    label=y_val,
    feature_names=features
)

# create a watchlist to monitor training and validation performance
watchlist = [
    (dtrain, 'train'),
    (dval, 'val')
]

print("Training:", dtrain.num_row(), dtrain.num_col())
print("Validation:", dval.num_row(), dval.num_col())

Training: 6000 16
Validation: 2000 16


In [15]:
# define XGBoost parameters
xgb_params = {
    'eta': 0.3,
    'max_depth': 6,
    'min_child_weight': 1,
    'objective': 'reg:squarederror',
    'nthread': 8,
    'seed': 1,
    'verbosity': 1,
}

# train XGBoost for 100 boosting rounds
model_xgb = xgb.train(
    xgb_params,
    dtrain,
    num_boost_round=100,
    evals=watchlist,
)

[0]	train-rmse:2.40653	val-rmse:2.46782
[1]	train-rmse:2.07098	val-rmse:2.15784
[2]	train-rmse:1.86454	val-rmse:1.97398
[3]	train-rmse:1.73360	val-rmse:1.86662
[4]	train-rmse:1.65059	val-rmse:1.80312
[5]	train-rmse:1.59328	val-rmse:1.76216
[6]	train-rmse:1.55435	val-rmse:1.74392
[7]	train-rmse:1.52677	val-rmse:1.73976
[8]	train-rmse:1.50265	val-rmse:1.73478
[9]	train-rmse:1.48761	val-rmse:1.73282
[10]	train-rmse:1.47125	val-rmse:1.73205
[11]	train-rmse:1.45902	val-rmse:1.73227
[12]	train-rmse:1.44829	val-rmse:1.73503
[13]	train-rmse:1.43604	val-rmse:1.73466
[14]	train-rmse:1.42972	val-rmse:1.73448
[15]	train-rmse:1.41908	val-rmse:1.73512
[16]	train-rmse:1.41206	val-rmse:1.73297
[17]	train-rmse:1.40940	val-rmse:1.73517
[18]	train-rmse:1.40278	val-rmse:1.73584
[19]	train-rmse:1.39637	val-rmse:1.73664
[20]	train-rmse:1.39368	val-rmse:1.73766
[21]	train-rmse:1.39254	val-rmse:1.73701
[22]	train-rmse:1.37639	val-rmse:1.74216
[23]	train-rmse:1.36470	val-rmse:1.74363
[24]	train-rmse:1.36071	va

In [16]:
# decrease the learning rate to 0.1
xgb_params['eta'] = 0.1

# train another XGBoost model for 100 boosting rounds
model_xgb_01 = xgb.train(
    xgb_params,
    dtrain,
    num_boost_round=100,
    evals=watchlist,
)

[0]	train-rmse:2.74739	val-rmse:2.78635
[1]	train-rmse:2.58527	val-rmse:2.63162
[2]	train-rmse:2.44540	val-rmse:2.50036
[3]	train-rmse:2.32187	val-rmse:2.38568
[4]	train-rmse:2.21445	val-rmse:2.28765
[5]	train-rmse:2.12109	val-rmse:2.20428
[6]	train-rmse:2.03996	val-rmse:2.13241
[7]	train-rmse:1.96795	val-rmse:2.06798
[8]	train-rmse:1.90640	val-rmse:2.01618
[9]	train-rmse:1.85240	val-rmse:1.96998
[10]	train-rmse:1.80592	val-rmse:1.93152
[11]	train-rmse:1.76410	val-rmse:1.89688
[12]	train-rmse:1.72781	val-rmse:1.86726
[13]	train-rmse:1.69564	val-rmse:1.84312
[14]	train-rmse:1.66745	val-rmse:1.82218
[15]	train-rmse:1.64286	val-rmse:1.80346
[16]	train-rmse:1.62187	val-rmse:1.79038
[17]	train-rmse:1.60321	val-rmse:1.77726
[18]	train-rmse:1.58694	val-rmse:1.76827
[19]	train-rmse:1.57240	val-rmse:1.75951
[20]	train-rmse:1.55847	val-rmse:1.75076
[21]	train-rmse:1.54640	val-rmse:1.74347
[22]	train-rmse:1.53643	val-rmse:1.73912
[23]	train-rmse:1.52675	val-rmse:1.73485
[24]	train-rmse:1.51763	va

**Answer: `0.1`**

With 100 boosting rounds, `eta=0.1` achieved a lower validation RMSE (1.725) than `eta=0.3` (1.826). Although the larger learning rate produced a lower training error, the smaller learning rate generalized better to the validation dataset.